In [9]:
import sys
import numpy as np
import pandas as pd
from copy import deepcopy
from SciExpeM_API.SciExpeM import SciExpeM
my_sciexpem = SciExpeM(username='YOUR_USERNAME', password='YOUR_PASSWORD')
# SciExpeM(username='YOUR_USERNAME', password='YOUR_PASSWORD', secure=False, verify=False, warning=False)
# database: porta 5432 all'indirizzo 127.0.0.1
from SciExpeM_API.Models import *

CONVERT_TO_BAR = {'atm': 1.01325, 'bar': 1., 'Torr': 0.00133322, 'Pa': 1e-5}

ID_ChemModels = [606, 618, 619, 621, 622, 623, 624, 625, 626, 627, 628, 629, 630, 631, 632, 633]
labels = ['NATPROT','2ndO2Add','Abs','betaC','betaH','Class5','ConcEl','CycEth','init','initH','ketoDec','ketoForm','LT_Iso','O2Add','QOOHDec','QOOHHO2']
exps = [3854]
conditions = ['43bar_692-1264K_phi1-2',]

for jj, j in enumerate(exps):
    exp = my_sciexpem.filterDatabase(model_name='Experiment', id = j)[0].experimental_data[0]
    exp['temperature [K]'] = 1000/exp['temperature [K]'].values
    exp = pd.DataFrame(exp[['temperature [K]', 'ignition delay [us]']].values, 
                       columns = ['1000/T [K]', 'IDT exp [us]' ])
    allexp = deepcopy(exp)
    print(exp)
    for i, ID_ChemModel in enumerate(ID_ChemModels):
        exec = my_sciexpem.filterDatabase(model_name='Execution', chemModel = ID_ChemModel, experiment = j)[0]
        x = 1000/exec.simulation_results[0]['ParametricAnalysisIDT']['T0'].values
        y = np.array(exec.simulation_results[0]['ParametricAnalysisIDT']['tau_P(slope)'].values)* 1e6
        if len(x) == len(exp['1000/T [K]']):
            sim = pd.DataFrame(np.array([y]).T * 1e6, columns=[ 'IDT [mus] ' + labels[i]])
        else:
            sim = pd.DataFrame(np.array([x, y]).T , columns=['1000/T [K]' , 'IDT [mus] ' + labels[i]])
        allexp = pd.concat([allexp,sim], axis=1)
    print(allexp)        
    with pd.ExcelWriter('IDTs2.xlsx', mode='a' ) as writer:
        allexp.to_excel(writer, sheet_name=conditions[jj])


Attention. SciExpeM is a singleton.
    1000/T [K]  IDT exp [us]
0     1.445087        3148.0
1     1.438849        2696.0
2     1.418440        1975.0
3     1.360544        1008.0
4     1.303781         680.0
5     1.300390         632.0
6     1.215067         376.0
7     1.196172         386.0
8     1.189061         311.0
9     1.136364         320.0
10    1.133787         334.0
11    1.218027         286.0
12    1.071811         408.0
13    1.067236         488.0
14    1.030928         483.0
15    1.016260         432.0
16    1.010101         394.0
17    0.970874         264.0
18    0.968992         319.0
19    0.961538         297.0
20    0.932836         218.0
21    0.802568          14.0
22    0.800641          22.0
23    0.791139          11.0
    1000/T [K]  IDT exp [us]    1000/T [K]  IDT [mus] NATPROT    1000/T [K]  \
0     1.445087        3148.0  1.445087e+06        6043.540986  1.445087e+06   
1     1.438849        2696.0  1.418440e+06        4063.460289  1.418440e+06   
2 